# Reward scale and shift (MountainCar)

`reward_scale` and `reward_shift` apply to the **step** reward before it is accumulated into `reward_episodic`. The raw environment reward is always available as `outputs[i]["reward"]` unchanged.

## `reward`, `reward_episodic`, and why they differ

Mouse-env exposes two reward signals in every output dict:

- **`reward`** — the raw environment reward (or `reset_reward` on reset frames). This is what `done=1` / `done=2` steps actually returned from Gymnasium.
- **`reward_episodic`** — a shaped, normalised signal intended for the model to train on. It is the cumulative sum of `(reward * reward_scale + reward_shift)` within the current episode, divided by `max_episode_steps`. Reset frames always emit `reward_episodic=0.0`.

**Why the two-signal design?** In continuing RL, credit must be able to pass across episode boundaries. A naive approach — just summing raw episodic rewards — conflates what the agent *did* within an episode (raw return) with the signal you want to train on (something that can propagate across resets). Keeping `reward` raw and exposing `reward_episodic` separately lets training code consume whatever framing it needs.

**`env.tracker.episode_cum_rewards[i]`** always reflects the raw (unscaled) return even when reward shaping is active, so evaluation numbers are always comparable.

## When to use shaping

MountainCar is a canonical example: the raw reward is `-1` every step until the goal is reached, which is sparse and hard to learn from. `reward_shift=1.0` flips the signal to `+1` per step (before normalisation), making the problem much easier for value-based methods. Use `reward_scale` to normalize the magnitude when combining envs with very different reward ranges.

In [ ]:
from mouse_envs import EnvConfig, make_env

## Config

In [ ]:
cfg = EnvConfig(
    id="MountainCar-v0",
    seed=0,
    num_envs=1,
    max_episode_steps=200,
    reward_scale=0.1,
    reward_shift=1.0,
)
env = make_env(cfg)

## Compare raw vs episodic reward

In [ ]:
for step in range(21):
    outputs = env.step(env.sample_random_inputs())
    r = outputs[0]
    print(
        f"step={step:2d}  raw_reward={r['reward'].item():.3f}  "
        f"episodic_reward={r['reward_episodic']:.3f}"
    )

env.close()